In [15]:
import os
import sys
import numpy as np
import moeabench as mb

# =======================================================
# 1. BLINDAGEM DE IMPORTAÇÃO
# =======================================================
DIR_ATUAL = os.path.abspath("")
if "src" not in os.listdir(DIR_ATUAL):
    DIR_RAIZ = os.path.dirname(DIR_ATUAL)
    if DIR_RAIZ not in sys.path: sys.path.insert(0, DIR_RAIZ)
else:
    if DIR_ATUAL not in sys.path: sys.path.insert(0, DIR_ATUAL)

from src.meamt import MEAMT


# =======================================================
# 2. CONFIGURAÇÃO
# =======================================================
MIN_TABLES_MEAMT = 30 
MAX_FES = 300000
NOBJ = 5

pop_meamt = MIN_TABLES_MEAMT * (2 ** NOBJ)
gen_meamt = MAX_FES // pop_meamt

exp1 = mb.experiment()
exp2 = mb.experiment()
exp3 = mb.experiment()

prob = mb.mops.DPF4(M=NOBJ, N=NOBJ + 20 - 1)
exp1.mop = prob
exp2.mop = prob
exp3.mop = prob

exp1.moea = mb.moeas.NSGA3(population=600, generations=500)
exp2.moea = MEAMT(population=pop_meamt, generations=gen_meamt)
exp3.moea = mb.moeas.NSGA2deap(population=300, generations=1000) 

# =======================================================
# 3. EXECUÇÃO
# =======================================================
print("Rodando NSGA-III...")
exp1.run(repeat=1, silent=True)

print("Rodando MEAMT...")
exp2.run(repeat=1, silent=True)

exp3.run()


# =======================================================
# 4. FILTRAGEM (DEVE OCORRER ANTES DAS MÉTRICAS!)
# =======================================================
run_meamt = exp2.runs[0]
F_atual = run_meamt._F_nd_history[-1] 

# O DTLZ3 tem raio 1. Se passou de 2, é lixo convergencial.
mascara_limite = np.all(F_atual <= 2.0, axis=1)

run_meamt._F_nd_history[-1] = F_atual[mascara_limite]


print(f"\n--- FILTRO DO MEAMT ---")
print(f"Total gerado: {len(F_atual)} pontos.")
print(f"Sobraram após filtro (<= 2.0): {np.sum(mascara_limite)} pontos.\n")



### Running **exp1**

Rodando NSGA-III...


### Running **exp2**

Rodando MEAMT...


### Running **exp3**

  Run 1/1:   0%|          | 0/1000 [00:00<?, ?gen/s]


--- FILTRO DO MEAMT ---
Total gerado: 635 pontos.
Sobraram após filtro (<= 2.0): 363 pontos.



In [16]:

# =======================================================
# 6. TOPOLOGIA
# =======================================================
mb.view.topology(exp1,show_gt=True)
mb.view.topology(exp2,show_gt=True) 
mb.view.topology(exp3,show_gt=True)
